In [1]:
import pandas as pd

In [10]:
raw = pd.read_csv("data\\vs_raw.csv")
ct = pd.read_csv("data\\sdtm_ct.csv")
output = pd.read_csv("data\\vs_golden.csv")


In [11]:
raw.head()

,STUDY,PATNUM,INSTANCE,FORM,FORML,VTLD,IT.HEIGHT_VSORRES,IT.WEIGHT,IT.TEMP,IT.TEMP_LOC,TMPTC,SYS_BP,DIA_BP,PULSE,SUBPOS
0,CDISCPILOT01,701-1015,Screening 1,VS,Vital Signs,26-Dec-2013,NaN,NaN,NaN,NaN,after Lying Down for 5 Minutes,131.0,64.0,57.0,SUPINE
1,CDISCPILOT01,701-1015,Screening 1,VS,Vital Signs,26-Dec-2013,NaN,NaN,NaN,NaN,after Standing for 1 Minute,129.0,83.0,62.0,STANDING
2,CDISCPILOT01,701-1015,Screening 1,VS,Vital Signs,26-Dec-2013,NaN,NaN,NaN,NaN,after Standing for 3 Minutes,147.0,57.0,65.0,STANDING
3,CDISCPILOT01,701-1015,Screening 1,VS,Vital Signs,26-Dec-2013,58.0,119.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CDISCPILOT01,701-1015,Screening 1,VS,Vital Signs,26-Dec-2013,NaN,NaN,96.9,ORAL CAVITY,NaN,NaN,NaN,NaN,NaN


In [22]:
ct.sample(10)

,codelist_code,term_code,term_value,collected_value,term_preferred_term,term_synonyms
47,C67153,C25347,Height,height,Height,NaN
3,C66726,C42887,AEROSOL,Aerosol,Aerosol Dosage Form,aer
129,VISITNUM,VISITNUM,3.5,Ambul ECG Placement,NaN,NaN
32,C66770,C44277,F,Degree Fahrenheit,Degree Fahrenheit,Degree Fahrenheit
118,VISITNUM,VISITNUM,4,Week 2,WEEK 2,Week2
102,ARM,ARM,Screen Failure,Screen Failure,NaN,NaN
144,VISIT,VISIT,WEEK 26,Week 26,WEEK 26,Week26
75,C99073,C25229,LEFT,Left,Left,NaN
19,C66741,C174446,TEMP,Body Temperature,Body Temperature,Body Temperature; Temperature
18,C66734,C49568,CM,Concomitant Medication Domain,Concomitant Medication Domain,Concomitant/Prior Medications


In [ ]:
ct['codelist_code'].unique()

<StringArray>
[  'C66726',   'C66728',   'C66729',   'C66734',   'C66741',   'C66742',
   'C66770',   'C66789',   'C67153',   'C71113',   'C71148',   'C71620',
   'C74456',   'C99073',      'TPT',   'TPTNUM', 'VISITNUM',    'VISIT',
   'C66781',   'C66731',   'C66790',   'C74457',      'ARM']
Length: 23, dtype: str

In [28]:
ct[ct['codelist_code'] == 'C71148']

,codelist_code,term_code,term_value,collected_value,term_preferred_term,term_synonyms
54,C71148,C111310,SEMI-RECUMBENT,Semi-Supine,Semi-Supine,Semi-Supine
55,C71148,C62122,SITTING,Sitting,Sitting,Sitting
56,C71148,C62165,PRONE,Prone,Prone,Prone
57,C71148,C62166,STANDING,Standing,Standing,Orthostatic; Standing
58,C71148,C62167,SUPINE,Supine,Supine,Supine


In [13]:
output.head()

,STUDYID,DOMAIN,USUBJID,VSSEQ,VSTESTCD,VSTEST,VSPOS,VSORRES,VSORRESU,VSSTRESC,...,VSBLFL,VISITNUM,VISIT,VISITDY,VSDTC,VSDY,VSTPT,VSTPTNUM,VSELTM,VSTPTREF
0,CDISCPILOT01,VS,01-701-1015,1,DIABP,Diastolic Blood Pressure,SUPINE,64.0,mmHg,64.0,...,NaN,1.0,SCREENING 1,-7.0,2013-12-26,-7,AFTER LYING DOWN FOR 5 MINUTES,815.0,PT5M,PATIENT SUPINE
1,CDISCPILOT01,VS,01-701-1015,2,DIABP,Diastolic Blood Pressure,STANDING,83.0,mmHg,83.0,...,NaN,1.0,SCREENING 1,-7.0,2013-12-26,-7,AFTER STANDING FOR 1 MINUTE,816.0,PT1M,PATIENT STANDING
2,CDISCPILOT01,VS,01-701-1015,3,DIABP,Diastolic Blood Pressure,STANDING,57.0,mmHg,57.0,...,NaN,1.0,SCREENING 1,-7.0,2013-12-26,-7,AFTER STANDING FOR 3 MINUTES,817.0,PT3M,PATIENT STANDING
3,CDISCPILOT01,VS,01-701-1015,4,DIABP,Diastolic Blood Pressure,SUPINE,68.0,mmHg,68.0,...,NaN,2.0,SCREENING 2,-1.0,2013-12-31,-2,AFTER LYING DOWN FOR 5 MINUTES,815.0,PT5M,PATIENT SUPINE
4,CDISCPILOT01,VS,01-701-1015,5,DIABP,Diastolic Blood Pressure,STANDING,59.0,mmHg,59.0,...,NaN,2.0,SCREENING 2,-1.0,2013-12-31,-2,AFTER STANDING FOR 1 MINUTE,816.0,PT1M,PATIENT STANDING


In [24]:
output.columns

Index(['STUDYID', 'DOMAIN', 'USUBJID', 'VSSEQ', 'VSTESTCD', 'VSTEST', 'VSPOS',
       'VSORRES', 'VSORRESU', 'VSSTRESC', 'VSSTRESN', 'VSSTRESU', 'VSSTAT',
       'VSLOC', 'VSBLFL', 'VISITNUM', 'VISIT', 'VISITDY', 'VSDTC', 'VSDY',
       'VSTPT', 'VSTPTNUM', 'VSELTM', 'VSTPTREF'],
      dtype='str')

In [29]:
# -----------------------------
# STEP 1: Convert wide → long
# -----------------------------
rows = []

for _, row in raw.iterrows():

    if pd.notna(row["SYS_BP"]):
        rows.append({
            "USUBJID": row["PATNUM"],
            "VSTESTCD": "SYSBP",
            "VSORRES": row["SYS_BP"],
            "VSPOS": row["SUBPOS"],
            "VISIT": row["INSTANCE"],
            "VSDTC": row["VTLD"],
            "STUDYID": row["STUDY"]
        })

    if pd.notna(row["DIA_BP"]):
        rows.append({
            "USUBJID": row["PATNUM"],
            "VSTESTCD": "DIABP",
            "VSORRES": row["DIA_BP"],
            "VSPOS": row["SUBPOS"],
            "VISIT": row["INSTANCE"],
            "VSDTC": row["VTLD"],
            "STUDYID": row["STUDY"]
        })

    if pd.notna(row["PULSE"]):
        rows.append({
            "USUBJID": row["PATNUM"],
            "VSTESTCD": "PULSE",
            "VSORRES": row["PULSE"],
            "VSPOS": row["SUBPOS"],
            "VISIT": row["INSTANCE"],
            "VSDTC": row["VTLD"],
            "STUDYID": row["STUDY"]
        })

    if pd.notna(row["IT.TEMP"]):
        rows.append({
            "USUBJID": row["PATNUM"],
            "VSTESTCD": "TEMP",
            "VSORRES": row["IT.TEMP"],
            "VSPOS": row["SUBPOS"],
            "VISIT": row["INSTANCE"],
            "VSDTC": row["VTLD"],
            "STUDYID": row["STUDY"]
        })

df = pd.DataFrame(rows)

# -----------------------------
# STEP 2: Fix USUBJID format
# -----------------------------
df["USUBJID"] = "01-" + df["USUBJID"].astype(str)

# -----------------------------
# STEP 3: Add DOMAIN
# -----------------------------
df["DOMAIN"] = "VS"

# -----------------------------
# STEP 4: Add VSTEST (full name)
# -----------------------------
test_map = {
    "SYSBP": "Systolic Blood Pressure",
    "DIABP": "Diastolic Blood Pressure",
    "PULSE": "Pulse Rate",
    "TEMP": "Temperature"
}

df["VSTEST"] = df["VSTESTCD"].map(test_map)

# -----------------------------
# STEP 5: Apply CT (Position)
# -----------------------------
def apply_ct(value, codelist_code):
    filtered = ct[ct["codelist_code"] == codelist_code]
    match = filtered[filtered["collected_value"] == value]
    if not match.empty:
        return match["term_value"].values[0]
    return value

df["VSPOS"] = df["VSPOS"].apply(lambda x: apply_ct(x, "C71148"))

# -----------------------------
# STEP 6: Add Units
# -----------------------------
def get_unit(test):
    if test in ["SYSBP", "DIABP"]:
        return apply_ct("mmHg", "C66770")
    elif test == "TEMP":
        return apply_ct("F", "C66770")
    elif test == "PULSE":
        return "beats/min"
    return ""

df["VSORRESU"] = df["VSTESTCD"].apply(get_unit)
df["VSSTRESU"] = df["VSORRESU"]

# -----------------------------
# STEP 7: Standard values
# -----------------------------
df["VSSTRESC"] = df["VSORRES"]

# -----------------------------
# STEP 8: Sequence number
# -----------------------------
df["VSSEQ"] = range(1, len(df)+1)

# -----------------------------
# STEP 9: Fix date format
# -----------------------------
df["VSDTC"] = pd.to_datetime(df["VSDTC"]).dt.strftime("%Y-%m-%d")

# -----------------------------
# STEP 10: Select final columns
# -----------------------------
final_cols = [
    "STUDYID","DOMAIN","USUBJID","VSSEQ",
    "VSTESTCD","VSTEST","VSPOS",
    "VSORRES","VSORRESU","VSSTRESC","VSSTRESU",
    "VISIT","VSDTC"
]

df = df[final_cols]


In [30]:
df.head()

,STUDYID,DOMAIN,USUBJID,VSSEQ,VSTESTCD,VSTEST,VSPOS,VSORRES,VSORRESU,VSSTRESC,VSSTRESU,VISIT,VSDTC
0,CDISCPILOT01,VS,01-701-1015,1,SYSBP,Systolic Blood Pressure,SUPINE,131.0,mmHg,131.0,mmHg,Screening 1,2013-12-26
1,CDISCPILOT01,VS,01-701-1015,2,DIABP,Diastolic Blood Pressure,SUPINE,64.0,mmHg,64.0,mmHg,Screening 1,2013-12-26
2,CDISCPILOT01,VS,01-701-1015,3,PULSE,Pulse Rate,SUPINE,57.0,beats/min,57.0,beats/min,Screening 1,2013-12-26
3,CDISCPILOT01,VS,01-701-1015,4,SYSBP,Systolic Blood Pressure,STANDING,129.0,mmHg,129.0,mmHg,Screening 1,2013-12-26
4,CDISCPILOT01,VS,01-701-1015,5,DIABP,Diastolic Blood Pressure,STANDING,83.0,mmHg,83.0,mmHg,Screening 1,2013-12-26


In [31]:
output.head()

,STUDYID,DOMAIN,USUBJID,VSSEQ,VSTESTCD,VSTEST,VSPOS,VSORRES,VSORRESU,VSSTRESC,...,VSBLFL,VISITNUM,VISIT,VISITDY,VSDTC,VSDY,VSTPT,VSTPTNUM,VSELTM,VSTPTREF
0,CDISCPILOT01,VS,01-701-1015,1,DIABP,Diastolic Blood Pressure,SUPINE,64.0,mmHg,64.0,...,NaN,1.0,SCREENING 1,-7.0,2013-12-26,-7,AFTER LYING DOWN FOR 5 MINUTES,815.0,PT5M,PATIENT SUPINE
1,CDISCPILOT01,VS,01-701-1015,2,DIABP,Diastolic Blood Pressure,STANDING,83.0,mmHg,83.0,...,NaN,1.0,SCREENING 1,-7.0,2013-12-26,-7,AFTER STANDING FOR 1 MINUTE,816.0,PT1M,PATIENT STANDING
2,CDISCPILOT01,VS,01-701-1015,3,DIABP,Diastolic Blood Pressure,STANDING,57.0,mmHg,57.0,...,NaN,1.0,SCREENING 1,-7.0,2013-12-26,-7,AFTER STANDING FOR 3 MINUTES,817.0,PT3M,PATIENT STANDING
3,CDISCPILOT01,VS,01-701-1015,4,DIABP,Diastolic Blood Pressure,SUPINE,68.0,mmHg,68.0,...,NaN,2.0,SCREENING 2,-1.0,2013-12-31,-2,AFTER LYING DOWN FOR 5 MINUTES,815.0,PT5M,PATIENT SUPINE
4,CDISCPILOT01,VS,01-701-1015,5,DIABP,Diastolic Blood Pressure,STANDING,59.0,mmHg,59.0,...,NaN,2.0,SCREENING 2,-1.0,2013-12-31,-2,AFTER STANDING FOR 1 MINUTE,816.0,PT1M,PATIENT STANDING


In [32]:
print(df.columns)

print("\n\n")

print(output.columns)

Index(['STUDYID', 'DOMAIN', 'USUBJID', 'VSSEQ', 'VSTESTCD', 'VSTEST', 'VSPOS',
       'VSORRES', 'VSORRESU', 'VSSTRESC', 'VSSTRESU', 'VISIT', 'VSDTC'],
      dtype='str')



Index(['STUDYID', 'DOMAIN', 'USUBJID', 'VSSEQ', 'VSTESTCD', 'VSTEST', 'VSPOS',
       'VSORRES', 'VSORRESU', 'VSSTRESC', 'VSSTRESN', 'VSSTRESU', 'VSSTAT',
       'VSLOC', 'VSBLFL', 'VISITNUM', 'VISIT', 'VISITDY', 'VSDTC', 'VSDY',
       'VSTPT', 'VSTPTNUM', 'VSELTM', 'VSTPTREF'],
      dtype='str')
